## 12. Cox Proportional Hazard Model

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

cox_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt',
    'annual_inc', 'revol_util', 'loan_to_income',
    'mort_acc', 'has_pub_rec', 'grade_woe', 'home_ownership_woe'
]

cox_df = df_cleaned[cox_features + ['default']].copy()
cox_df['duration_months'] = survival_df['duration_months']
cox_df = cox_df.dropna()
cox_sample = cox_df.sample(n=100000, random_state=42)  # sample for speed

print("Fitting Cox model...")
cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_sample, duration_col='duration_months', event_col='default', show_progress=True)
cph.print_summary()

# Hazard ratio plot
fig, ax = plt.subplots(figsize=(10, 7))
cph.plot(ax=ax)
ax.set_title('Cox Model — Hazard Ratios with 95% CI')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Proportional hazard assumption test
print("\n--- PH Assumption Test (Schoenfeld Residuals) ---")
ph_test = proportional_hazard_test(cph, cox_sample, time_transform='rank')
ph_test.print_summary()

# Sample borrower lifetime PD curves
sample_borrowers = pd.DataFrame({
    'int_rate':           [6.0,  11.0,  17.0,  25.0],
    'fico_range_low':     [780,   720,   670,   640],
    'dti':                [8.0,  15.0,  25.0,  35.0],
    'loan_amnt':          [5000, 12000, 20000, 30000],
    'annual_inc':         [90000, 65000, 45000, 35000],
    'revol_util':         [10.0,  35.0,  60.0,  80.0],
    'loan_to_income':     [0.06,  0.18,  0.44,  0.86],
    'mort_acc':           [3,     1,     0,     0],
    'has_pub_rec':        [0,     0,     0,     1],
    'grade_woe':          [1.355, 0.479, -0.559, -1.386],
    'home_ownership_woe': [0.182, 0.182, -0.192, -0.192],
})
labels = ['Very Low Risk (A)', 'Low Risk (B)', 'Medium Risk (D)', 'High Risk (G)']
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
sf = cph.predict_survival_function(sample_borrowers)

fig, ax = plt.subplots(figsize=(12, 7))
for i, (label, color) in enumerate(zip(labels, colors)):
    ax.plot(sf.index, sf.iloc[:,i], label=label, color=color, linewidth=2)
ax.set_title('Cox Model — Individual Lifetime PD Curves')
ax.set_xlabel('Months Since Origination')
ax.set_ylabel('Survival Probability')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

# 12/24/36m PD
print(f"\n{'Borrower':<30} {'12m PD':>10} {'24m PD':>10} {'36m PD':>10}")
for i, label in enumerate(labels):
    s = sf.iloc[:,i]
    pd12 = 1 - s.iloc[s.index.get_indexer([12], method='nearest')[0]]
    pd24 = 1 - s.iloc[s.index.get_indexer([24], method='nearest')[0]]
    pd36 = 1 - s.iloc[s.index.get_indexer([36], method='nearest')[0]]
    print(f"{label:<30} {pd12:>10.4f} {pd24:>10.4f} {pd36:>10.4f}")

**Note:** Six features violate the proportional hazard assumption. This is expected in credit risk — high-risk borrowers default early, leaving a stronger survivor population. For production use, a time-varying coefficient or stratified Cox model would address this.